In [3]:
# IMPORTANT: after Cell 1, restart runtime once, then run from Cell 2

import os, json, re, gc, torch, pandas as pd
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from sklearn.metrics import accuracy_score, f1_score
import evaluate

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [4]:
def pick_file(candidates, name):
    for p in candidates:
        if Path(p).exists():
            return str(Path(p))
    raise FileNotFoundError(f"{name} not found. Checked: {candidates}")

TRAIN_FILE = pick_file(["/content/train.jsonl", "train.jsonl"], "train.jsonl")
VAL_FILE   = pick_file(["/content/val.jsonl", "val.jsonl"], "val.jsonl")
TEST_FILE  = pick_file(["/content/test.jsonl", "test.jsonl"], "test.jsonl")

raw = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE, "test": TEST_FILE}
)

print("Loaded:", {k: len(v) for k, v in raw.items()})

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loaded: {'train': 153, 'validation': 32, 'test': 32}


In [5]:
def build_text(ex):
    caption = (ex.get("input", {}) or {}).get("image_caption") or ex.get("refined_caption", "")
    voice = ex.get("voice_text") or (ex.get("input", {}) or {}).get("voice_text", "")

    category = ex.get("category", "")
    severity = str(ex.get("severity", "")).lower()
    department = (ex.get("routing", {}) or {}).get("primary_department") or ex.get("department", "")
    description = ex.get("complaint_description", "")

    target = {
        "category": category,
        "severity": severity,
        "department": department,
        "complaint_description": description
    }

    ex["caption_used"] = caption
    ex["voice_used"] = voice
    ex["target_json"] = json.dumps(target, ensure_ascii=False)

    ex["text"] = (
        "### Instruction:\n"
        "Analyze hospital complaint input and return ONLY valid JSON with keys: "
        "category, severity, department, complaint_description.\n\n"
        f"### Input:\nCaption: {caption}\nVoice: {voice}\n\n"
        "### Response:\n"
        f"{ex['target_json']}"
    )
    return ex

dataset = raw.map(build_text)
print(dataset["train"][0]["text"][:700])

Map:   0%|          | 0/153 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

### Instruction:
Analyze hospital complaint input and return ONLY valid JSON with keys: category, severity, department, complaint_description.

### Input:
Caption: The hospital bed appears to be damaged, with visible signs of wear and tear. The bed frame is broken, indicating a need for immediate repair or replacement.
Voice: The bed in my hospital room is broken, it's really uncomfortable and I'm worried about my safety.

### Response:
{"category": "Broken Hospital Bed", "severity": "high", "department": "Maintenance", "complaint_description": "I was shocked to see the state of the hospital bed in my room, it's clearly broken and looks like it hasn't been maintained in a long time. As a pat


In [6]:
# OPTION A (recommended first): Mistral baseline-aligned fine-tune
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
OUTPUT_TAG = "mistral7b_instruct_v02"

# OPTION B (run in separate run/runtime): Qwen
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# OUTPUT_TAG = "qwen2_5_1_5b_instruct"

print("MODEL_ID:", MODEL_ID)

MODEL_ID: mistralai/Mistral-7B-Instruct-v0.2


In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

print("Model + tokenizer loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model + tokenizer loaded.


In [8]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758499550960753


In [9]:
training_args = TrainingArguments(
    output_dir=f"/content/{OUTPUT_TAG}_ckpt",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none",
    gradient_checkpointing=True
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=768,
    tokenizer=tokenizer,
    packing=False
)

print("Trainer ready.")

Map:   0%|          | 0/153 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Trainer ready.


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [10]:
train_result = trainer.train()
print("Training done.")

ADAPTER_DIR = f"/content/{OUTPUT_TAG}_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter:", ADAPTER_DIR)

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Step,Training Loss,Validation Loss
25,0.387700,0.340045
50,0.119600,0.305742


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Training done.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved adapter: /content/mistral7b_instruct_v02_adapter


In [11]:
VALID_CATEGORIES = sorted(set(dataset["train"]["category"] + dataset["validation"]["category"]))
VALID_SEVERITIES = sorted(set([str(x).lower() for x in dataset["train"]["severity"] + dataset["validation"]["severity"]]))

def get_primary_dept(ex):
    return (ex.get("routing", {}) or {}).get("primary_department") or ex.get("department", "")

VALID_DEPARTMENTS = sorted(set([get_primary_dept(x) for x in dataset["train"]] + [get_primary_dept(x) for x in dataset["validation"]]))

def parse_json_output(raw_text):
    if not raw_text:
        return {}, False
    cleaned = re.sub(r"```(?:json)?", "", raw_text, flags=re.IGNORECASE).replace("```", "").strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = cleaned[start:end+1]
        try:
            return json.loads(candidate), True
        except json.JSONDecodeError:
            pass
    try:
        return json.loads(cleaned), True
    except json.JSONDecodeError:
        return {}, False

def infer_one(ex):
    prompt = (
        "### Instruction:\n"
        "Analyze hospital complaint input and return ONLY valid JSON with keys: "
        "category, severity, department, complaint_description.\n\n"
        f"### Input:\nCaption: {ex['caption_used']}\nVoice: {ex['voice_used']}\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    resp = decoded.split("### Response:")[-1].strip()
    parsed, ok = parse_json_output(resp)
    return parsed, ok, resp

In [12]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

rows = []
for ex in dataset["test"]:
    gt_cat = ex.get("category", "")
    gt_sev = str(ex.get("severity", "")).lower()
    gt_dep = get_primary_dept(ex)
    gt_desc = ex.get("complaint_description", "")

    pred, json_ok, raw_resp = infer_one(ex)
    pred_cat = str(pred.get("category", "")).strip()
    pred_sev = str(pred.get("severity", "")).strip().lower()
    pred_dep = str(pred.get("department", "")).strip()
    pred_desc = str(pred.get("complaint_description", "")).strip()

    rows.append({
        "image_id": ex.get("image_id", ""),
        "gt_category": gt_cat, "pred_category": pred_cat,
        "gt_severity": gt_sev, "pred_severity": pred_sev,
        "gt_department": gt_dep, "pred_department": pred_dep,
        "gt_description": gt_desc, "pred_description": pred_desc,
        "json_valid": json_ok,
        "raw_output": raw_resp
    })

df = pd.DataFrame(rows)

cat_acc = accuracy_score(df["gt_category"], df["pred_category"])
cat_f1 = f1_score(df["gt_category"], df["pred_category"], average="macro", zero_division=0)

sev_acc = accuracy_score(df["gt_severity"], df["pred_severity"])
sev_f1 = f1_score(df["gt_severity"], df["pred_severity"], average="macro", zero_division=0)

dep_acc = accuracy_score(df["gt_department"], df["pred_department"])
dep_f1 = f1_score(df["gt_department"], df["pred_department"], average="macro", zero_division=0)

pred_desc = df["pred_description"].fillna("").tolist()
gt_desc = df["gt_description"].fillna("").tolist()

rouge_res = rouge.compute(predictions=pred_desc, references=gt_desc)
bleu_res = bleu.compute(predictions=pred_desc, references=[[x] for x in gt_desc])

metrics = {
    "model_id": MODEL_ID,
    "records": len(df),
    "json_validity_rate": float(df["json_valid"].mean()),
    "category_accuracy": float(cat_acc),
    "category_macro_f1": float(cat_f1),
    "severity_accuracy": float(sev_acc),
    "severity_macro_f1": float(sev_f1),
    "department_accuracy": float(dep_acc),
    "department_macro_f1": float(dep_f1),
    "bleu": float(bleu_res["bleu"]),
    "rouge1": float(rouge_res["rouge1"]),
    "rouge2": float(rouge_res["rouge2"]),
    "rougeL": float(rouge_res["rougeL"]),
}

OUT_DIR = f"/content/finetune_outputs_{OUTPUT_TAG}"
os.makedirs(OUT_DIR, exist_ok=True)

df.to_csv(f"{OUT_DIR}/finetuned_test_predictions.csv", index=False)
with open(f"{OUT_DIR}/finetuned_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved:")
print(f"- {OUT_DIR}/finetuned_test_predictions.csv")
print(f"- {OUT_DIR}/finetuned_metrics.json")
print("\nMetrics:")
print(json.dumps(metrics, indent=2))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Saved:
- /content/finetune_outputs_mistral7b_instruct_v02/finetuned_test_predictions.csv
- /content/finetune_outputs_mistral7b_instruct_v02/finetuned_metrics.json

Metrics:
{
  "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
  "records": 32,
  "json_validity_rate": 1.0,
  "category_accuracy": 1.0,
  "category_macro_f1": 1.0,
  "severity_accuracy": 1.0,
  "severity_macro_f1": 1.0,
  "department_accuracy": 1.0,
  "department_macro_f1": 1.0,
  "bleu": 0.479794803807503,
  "rouge1": 0.6740647713802905,
  "rouge2": 0.510173046934123,
  "rougeL": 0.6069189062786737
}


In [1]:
# 1. Uninstall any existing versions to start clean
!pip uninstall -y bitsandbytes accelerator transformers peft trl

# 2. Install specific compatible versions
# bitsandbytes >= 0.43.1 is required to fix the Triton/Python 3.12 error
!pip install --quiet bitsandbytes>=0.43.1
!pip install --quiet transformers==4.40.0
!pip install --quiet accelerate==0.29.0
!pip install --quiet peft==0.10.0
!pip install --quiet trl==0.8.6
!pip install --quiet datasets==2.18.0
!pip install --quiet evaluate rouge_score

# 3. Force update Triton (this is the specific fix for your error)
!pip install -U --quiet triton

# 4. Import libraries
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import pandas as pd

print("Setup Complete! Libraries imported successfully.")

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 7.9 MB/s eta 